# County Rainfall Analysis

In [1]:
import csv
import numpy as np

months = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 1. Clean Machakos

In [2]:
def clean_row(raw):
    vals, bad = [], []
    for i, x in enumerate(raw):
        try:
            vals.append(int(x))
        except ValueError:
            vals.append(None)
            bad.append(i)
    avg = round(np.mean([v for v in vals if v is not None]))
    return [avg if v is None else v for v in vals], bad, avg

machakos, bad, avg = clean_row(machakos_raw)
print("Imputed months:", [months[i] for i in bad], "with", avg, "mm")


Imputed months: ['April', 'September'] with 68 mm


## 2. Combine into one array

In [3]:
R = np.array(clean_rows + [machakos])
print(R.shape, R.dtype)

(5, 12) int64


## 3. Totals, averages

In [4]:
totals = R.sum(axis=1)
print(dict(zip(counties, totals.tolist())))
print(dict(zip(months, R.mean(axis=0).round(1).tolist())))

{'Nairobi': 1045, 'Nakuru': 916, 'Kisumu': 1371, 'Mombasa': 1075, 'Machakos': 811}
{'January': 47.0, 'February': 47.6, 'March': 91.0, 'April': 164.4, 'May': 171.2, 'June': 67.6, 'July': 56.6, 'August': 58.2, 'September': 64.2, 'October': 67.8, 'November': 127.4, 'December': 80.6}


## 4. Wettest month and dry months

In [5]:
wettest = np.array(months)[R.argmax(axis=1)]
for c, m in zip(counties, wettest):
    print(c, "->", m)
dry = (R < 50).sum(axis=1)
print(dict(zip(counties, dry.tolist())))

Nairobi -> April
Nakuru -> April
Kisumu -> April
Mombasa -> May
Machakos -> November
{'Nairobi': 4, 'Nakuru': 3, 'Kisumu': 0, 'Mombasa': 2, 'Machakos': 6}


## 5. Planting windows

In [6]:
S = R[:, :-1] + R[:, 1:]
k = S.argmax(axis=1)
mo = np.array(months)
windows = np.char.add(np.char.add(mo[k], "-"), mo[k + 1])
for c, w, s in zip(counties, windows, S.max(axis=1)):
    print(c, w, s, "mm")

Nairobi April-May 395 mm
Nakuru April-May 265 mm
Kisumu April-May 420 mm
Mombasa April-May 390 mm
Machakos November-December 278 mm


## 6. Write CSV

In [7]:
with open("rainfall_summary.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    w.writerows(zip(counties, totals.tolist(), wettest.tolist(), dry.tolist(), windows.tolist()))

## Planting Advisory
Nairobi, Nakuru, Kisumu and Mombasa should plant in April-May, when two-month rainfall is highest (265-420 mm). Machakos should plant in November-December, as its long rains are weak. Kisumu is the wettest county (1371 mm a year) with no dry months. Machakos has six dry months, so farmers there should pick drought-tolerant seed and conserve water. Send SMS alerts a week before each window.

## Data Quality Notes
Machakos came from hand-typed text. April and September were invalid ("N/A" and blank) and were imputed with the mean of the 10 valid values (68 mm), so Machakos totals and dry months are estimates. The padded value " 20 " was kept. Verify with the station.